In [1]:
using Random
#Load your library of functions
include("utils.2.0.jl")
# Set a global random seed for reproducibility
Random.seed!(42)


TaskLocalRNG()

# Load packages

In [ ]:
# skip reinstalling packages we already have
using Pkg
using Plots
pkgs = [
    "MLJ", "MLJBase", "MLJModels", "MLJEnsembles", "MLJLinearModels",
    "DecisionTree", "MLJDecisionTreeInterface", "NaiveBayes", 
    "MLJNaiveBayesInterface", "EvoTrees", "CategoricalArrays", "Random",
    "LIBSVM", "MLJLIBSVMInterface", "Plots", "MLJModelInterface",
    "CSV", "DataFrames", "UrlDownload", "XGBoost"
]

# Filter out packages already installed
missing_pkgs = filter(pkg -> !(pkg in keys(Pkg.project().dependencies)), pkgs)

if !isempty(missing_pkgs)
    println("Installing missing packages: ", missing_pkgs)
    Pkg.add(missing_pkgs)
else
    println(" All required packages are already installed.")
end


 All required packages are already installed.


# Load Data

In [19]:
using CSV, DataFrames, Random
using CategoricalArrays

df = CSV.read("./data/updated_pollution_dataset.csv", DataFrame)

# Some log
println("First 5 rows of df:")
show(df[1:5, :], allcols=true)


# Convert column 10 to categorical (in-place!)
df[!, 10] = categorical(df[!, 10])

# Extract the integer codes of the categories
targets = Float32.(levelcode.(df[!, 10]))

inputs  = Matrix{Float32}(df[:, 1:9])

N = size(df, 1)
#trainIdx, valIdx, testIdx = holdOut(N, 0.3, 0.3)
trainIdx, valIdx = holdOut(N, 0.3)

trainingInputs  = inputs[trainIdx, :]
valInputs       = inputs[valIdx, :]
#testInputs      = inputs[testIdx, :]

trainingTargets = targets[trainIdx]
valTargets      = targets[valIdx]
#testTargets     = targets[testIdx]


println("\n\nFirst 5 targets:")
println(targets[1:5])

println("Training inputs (first 5 rows):")
for i in 1:5
    println(trainingInputs[i, :])
end



println("\n\n=========== Normalizing Inputs ===========")

# Compute normalization parameters from TRAINING set only
normParams = calculateMinMaxNormalizationParameters(trainingInputs)

# Normalize training set IN PLACE
normalizeMinMax!(trainingInputs, normParams)

# Normalize validation set -> returns new array
valInputs_normalized = normalizeMinMax(valInputs, normParams)

println("\nTraining inputs after normalization (first 5 rows):")
for i in 1:5
    println(trainingInputs[i, :])
end

println("\nValidation inputs after normalization (first 5 rows):")
for i in 1:5
    println(valInputs_normalized[i, :])
end

First 5 rows of df:
5×10 DataFrame
 Row │ Temperature  Humidity  PM2.5    PM10     NO2      SO2      CO       Proximity_to_Industrial_Areas  Population_Density  Air Quality 
     │ Float64      Float64   Float64  Float64  Float64  Float64  Float64  Float64                        Int64               String15    
─────┼────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
   1 │        29.8      59.1      5.2     17.9     18.9      9.2     1.72                            6.3                 319  Moderate
   2 │        28.3      75.6      2.3     12.2     30.8      9.7     1.64                            6.0                 611  Moderate
   3 │        23.1      74.7     26.7     33.8     24.4     12.6     1.63                            5.2                 619  Moderate
   4 │        27.1      39.1      6.1      6.3     13.5      5.3     1.15                           11.1                 551  Good
   5 │      

# SVM

In [ ]:
# define the model
using LIBSVM
using MLJ;
SVMClassifier = MLJ.@load SVC pkg=LIBSVM verbosity=0
model = SVMClassifier(kernel=LIBSVM.Kernel.RadialBasis, cost=1.0, gamma=2.0, degree=Int32(3))
# create the machine object
mach = machine(model, MLJ.table(trainingInputs), categorical(trainingTargets))
#train
MLJ.fit!(mach, verbosity=0)



println("valTargets:")
display(valTargets[1:5])
# println("testOutputs:")
# display(testOutputs[1:5])


# predict
y_pred = MLJ.predict(mach, MLJ.table(valInputs));
accuracy, error_rate, recall, spec, ppv, npv, f1, cm =confusionMatrix(string.(y_pred), string.(valTargets))
# Pretty printing
println("\n================= MODEL PERFORMANCE =================")
println(" Accuracy        : ", round(accuracy,    digits=4))
println(" Error Rate      : ", round(error_rate,  digits=4))
println(" Recall (Sens.)  : ", round(recall,      digits=4))
println(" Specificity     : ", round(spec,        digits=4))
println(" PPV (Precision) : ", round(ppv,         digits=4))
println(" NPV             : ", round(npv,         digits=4))
println(" F1 Score        : ", round(f1,          digits=4))
println("=====================================================\n")

println("Confusion Matrix:")
display(cm)

# DT

In [10]:
DTClassifier = MLJ.@load DecisionTreeClassifier pkg=DecisionTree verbosity=0
model = DTClassifier(max_depth=4, rng=Random.MersenneTwister(1))
mach = machine(model, MLJ.table(trainingInputs), categorical(trainingTargets))
#train
MLJ.fit!(mach, verbosity=0)



println("valTargets:")
display(valTargets[1:5])


# predict
y_pred = MLJ.predict(mach, MLJ.table(valInputs));
accuracy, error_rate, recall, spec, ppv, npv, f1, cm =confusionMatrix(string.(y_pred), string.(valTargets))
# Pretty printing
println("\n================= MODEL PERFORMANCE =================")
println(" Accuracy        : ", round(accuracy,    digits=4))
println(" Error Rate      : ", round(error_rate,  digits=4))
println(" Recall (Sens.)  : ", round(recall,      digits=4))
println(" Specificity     : ", round(spec,        digits=4))
println(" PPV (Precision) : ", round(ppv,         digits=4))
println(" NPV             : ", round(npv,         digits=4))
println(" F1 Score        : ", round(f1,          digits=4))
println("=====================================================\n")

println("Confusion Matrix:")
display(cm)

valTargets:

================= MODEL PERFORMANCE =================
 Accuracy        : 0.0
 Error Rate      : 1.0
 Recall (Sens.)  : 0.0
 Specificity     : 1.0
 PPV (Precision) : 0.0
 NPV             : 0.7038
 F1 Score        : 0.0

Confusion Matrix:


5-element Vector{Float32}:
 1.0
 4.0
 3.0
 1.0
 3.0

5×5 Matrix{Int64}:
 0  0  0  0  592
 0  0  0  0  301
 0  0  0  0  447
 0  0  0  0  160
 0  0  0  0    0

# KNN

In [11]:
kNNClassifier = MLJ.@load KNNClassifier pkg=NearestNeighborModels verbosity=0
model = kNNClassifier(K=3)
mach = machine(model, MLJ.table(trainingInputs), categorical(trainingTargets))
#train
MLJ.fit!(mach, verbosity=0)



# println("testTargets:")
# display(testTargets[1:5])
# println("testOutputs:")
# display(testOutputs[1:5])


# predict
y_pred = MLJ.predict(mach, MLJ.table(valInputs));
accuracy, error_rate, recall, spec, ppv, npv, f1, cm =confusionMatrix(string.(y_pred), string.(valTargets))
# Pretty printing
println("\n================= MODEL PERFORMANCE =================")
println(" Accuracy        : ", round(accuracy,    digits=4))
println(" Error Rate      : ", round(error_rate,  digits=4))
println(" Recall (Sens.)  : ", round(recall,      digits=4))
println(" Specificity     : ", round(spec,        digits=4))
println(" PPV (Precision) : ", round(ppv,         digits=4))
println(" NPV             : ", round(npv,         digits=4))
println(" F1 Score        : ", round(f1,          digits=4))
println("=====================================================\n")

println("Confusion Matrix:")
display(cm)

5×5 Matrix{Int64}:
 0  0  0  0  592
 0  0  0  0  301
 0  0  0  0  447
 0  0  0  0  160
 0  0  0  0    0


================= MODEL PERFORMANCE =================
 Accuracy        : 0.0
 Error Rate      : 1.0
 Recall (Sens.)  : 0.0
 Specificity     : 1.0
 PPV (Precision) : 0.0
 NPV             : 0.7038
 F1 Score        : 0.0

Confusion Matrix:


# ANN

In [ ]:

using Flux: onehotbatch, crossentropy, softmax, params

function trainClassANN(
    topology::Vector{Int},
    trainingDataset::Tuple{Matrix{Float32}, Vector{Int}}; 
    validationDataset=(Array{Float32,2}(undef,0,0), Int[]),
    transferFunctions::Union{Nothing, Vector{Function}}=nothing,
    maxEpochs::Int=1000,
    learningRate::Float32=0.01,
    showText::Bool=false
)

    # Unpack datasets
    X_train, y_train_int = trainingDataset
    X_val,   y_val_int   = validationDataset

    # --- FIX ORIENTATION HERE ---
    if size(X_train, 1) < size(X_train, 2)
        X_train = X_train'
    end
    if size(X_val, 1) < size(X_val, 2)
        X_val = X_val'
    end
    # --------------------------------

    # Number of classes
    n_classes = maximum(y_train_int)

    # One-hot encode labels
    Y_train = onehotbatch(y_train_int, 1:n_classes)
    Y_val   = length(y_val_int) > 0 ? onehotbatch(y_val_int, 1:n_classes) : nothing

    # Build topology
    topology_new = copy(topology)
    topology_new[end] = n_classes

    # Transfer functions
    if transferFunctions === nothing
        transferFunctions = vcat([σ for _ in 1:length(topology_new)-1], [identity])
    elseif length(transferFunctions) != length(topology_new)
        error("transferFunctions length must match topology length")
    end

    # Build model
    layers = []
    input_size = topology_new[1]

    for i in 2:length(topology_new)
        push!(layers, Dense(input_size, topology_new[i], transferFunctions[i]))
        input_size = topology_new[i]
    end

    model = Chain(layers..., softmax)

    # Optimizer
    opt = Descent(learningRate)

    loss() = crossentropy(model(X_train), Y_train)

    for epoch in 1:maxEpochs
        gs = gradient(() -> loss(), params(model))
        Flux.Optimise.update!(opt, params(model), gs)

        if showText && epoch % 10 == 0
            println("Epoch $epoch, Loss: ", loss())
        end
    end

    return model
end



# ---------------------------------------
# Example usage
# ---------------------------------------
trainingInputs = trainingInputs'
valInputs      = valInputs'

topology = [size(trainingInputs, 1), 16, 8, 1]
model = trainClassANN(
    topology,          # topology (positional)
    (trainingInputs, Int.(trainingTargets));      # trainingDataset (positional)
    validationDataset=(valInputs, Int.(valTargets)),
    maxEpochs=100,
    learningRate=0.01f0,
    showText=true
)


# ANN

## Load data

In [59]:
using CSV, DataFrames, Random
using CategoricalArrays

df = CSV.read("./data/updated_pollution_dataset.csv", DataFrame)

# Some log
println("First 5 rows of df:")
show(df[1:5, :], allcols=true)


# Convert column 10 to categorical (in-place!)
df[!, 10] = categorical(df[!, 10])

# Extract the integer codes of the categories
targets = Float32.(levelcode.(df[!, 10]))

inputs  = Matrix{Float32}(df[:, 1:9])

N = size(df, 1)
trainIdx, valIdx, testIdx = holdOut(N, 0.3, 0.3)
println("Train indices: ", length(trainIdx))
println("Validation indices: ", length(valIdx))
println("Test indices: ", length(testIdx))
println("Dataset size: ", size(dataset))

trainingInputs  = inputs[trainIdx, :]
valInputs       = inputs[valIdx, :]
testInputs      = inputs[testIdx, :]

trainingTargets = targets[trainIdx]
valTargets      = targets[valIdx]
testTargets     = targets[testIdx]


println("\n\nFirst 5 targets:")
println(targets[1:5])

println("Training inputs (first 5 rows):")
for i in 1:5
    println(trainingInputs[i, :])
end


println("\n\n=========== Normalizing Inputs ===========")

# Compute normalization parameters from TRAINING set only
normParams = calculateMinMaxNormalizationParameters(trainingInputs)

# Normalize training set IN PLACE
normalizeMinMax!(trainingInputs, normParams)

# Normalize validation set in place
#valInputs_normalized = normalizeMinMax(valInputs, normParams)
normalizeMinMax!(valInputs, normParams)

# Normalize test set in place
#valInputs_normalized = normalizeMinMax(valInputs, normParams)
normalizeMinMax!(testInputs, normParams)


println("\nTraining inputs after normalization (first 5 rows):")
for i in 1:5
    println(trainingInputs[i, :])
end

println("\nValidation inputs after normalization (first 5 rows):")
for i in 1:5
    println(valInputs[i, :])
end

println("\nTest inputs after normalization (first 5 rows):")
for i in 1:5
    println(testInputs[i, :])
end


# Convert to float32 for Flux compatibility
trainingInputs = Float32.(trainingInputs)
valInputs = Float32.(valInputs)
testInputs = Float32.(testInputs)

# Clip values to [0,1] after normalization
valInputs .= clamp.(valInputs, 0f0, 1f0)
testInputs .= clamp.(testInputs, 0f0, 1f0)


# Values should only fall within [0,1]
@assert(all(minimum(trainingInputs, dims=1) .== 0))
@assert(all(maximum(trainingInputs, dims=1) .== 1))
@assert(all(minimum(valInputs, dims=1) .>= 0))
@assert(all(maximum(valInputs, dims=1) .<= 1))
@assert(all(minimum(testInputs, dims=1) .>= 0))
@assert(all(maximum(testInputs, dims=1) .<= 1))


println("Train inputs range per feature: ", (minimum(trainingInputs, dims=1), maximum(trainingInputs, dims=1)))
println("Validation inputs range per feature: ", (minimum(valInputs, dims=1), maximum(valInputs, dims=1)))
println("Test inputs range per feature: ", (minimum(testInputs, dims=1), maximum(testInputs, dims=1)))


First 5 rows of df:
5×10 DataFrame
 Row │ Temperature  Humidity  PM2.5    PM10     NO2      SO2      CO       Proximity_to_Industrial_Areas  Population_Density  Air Quality 
     │ Float64      Float64   Float64  Float64  Float64  Float64  Float64  Float64                        Int64               String15    
─────┼────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
   1 │        29.8      59.1      5.2     17.9     18.9      9.2     1.72                            6.3                 319  Moderate
   2 │        28.3      75.6      2.3     12.2     30.8      9.7     1.64                            6.0                 611  Moderate
   3 │        23.1      74.7     26.7     33.8     24.4     12.6     1.63                            5.2                 619  Moderate
   4 │        27.1      39.1      6.1      6.3     13.5      5.3     1.15                           11.1                 551  Good
   5 │      

# ANN

In [66]:
function trainClassANN(topology::AbstractArray{<:Int,1},  
            trainingDataset::Tuple{AbstractArray{<:Real,2}, AbstractArray{Bool,2}}; 
            # --- Requirement: optional validation dataset with default empty arrays ---
            validationDataset::Tuple{AbstractArray{<:Real,2}, AbstractArray{Bool,2}}= 
                    (Array{eltype(trainingDataset[1]),2}(undef,0,0), falses(0,0)), 
            # --- Requirement: optional test dataset with default empty arrays ---
            testDataset::Tuple{AbstractArray{<:Real,2}, AbstractArray{Bool,2}}= 
                    (Array{eltype(trainingDataset[1]),2}(undef,0,0), falses(0,0)), 
            transferFunctions::AbstractArray{<:Function,1}=fill(σ, length(topology)), 
            maxEpochs::Int=1000, minLoss::Real=0.0, learningRate::Real=0.01,  
            # --- Requirement: maxEpochsVal parameter (early stopping patience), default 20 ---
            maxEpochsVal::Int=20, showText::Bool=false) 

    # --- Unpacking datasets ---
    (inputs, targets) = trainingDataset
    (val_inputs, val_targets) = validationDataset
    (test_inputs, test_targets) = testDataset

    # --- Ensures dataset dimensions match ---
    @assert size(inputs,1) == size(targets,1)
    @assert size(val_inputs,1) == size(val_targets,1)
    @assert size(test_inputs,1) == size(test_targets,1)

    # --- Requirement: build ANN with given topology ---
    ann = buildClassANN(size(inputs,2), topology, size(targets,2))

    # --- Define loss function (binary or multi-class) ---
    # discriminates based on the number of output neurons
    loss(model,x,y) = (size(y,1) == 1) ? Losses.binarycrossentropy(model(x),y) : Losses.crossentropy(model(x),y)

    # --- Requirement: loss histories for training/validation/test ---
    trainingLosses = Float32[]
    validationLosses = Float32[]
    testLosses = Float32[]

    # --- Initial losses (cycle 0, before training) ---
    numEpoch = 0
    trainingLoss = loss(ann, inputs', targets')
    push!(trainingLosses, trainingLoss)
    
    # init message buffer
    log_message = []
    log_message = "Epoch $numEpoch - loss: $(round(trainingLoss, digits=4))"


    # --- if validation set is provided ---                          
    if size(val_inputs,1) > 0
        validationLoss = loss(ann, val_inputs', val_targets')
        push!(validationLosses, validationLoss)

        # update message buffer
        log_message *= " - val_loss: $(round(validationLoss, digits=4))"
    end

     # --- if test set is provided ---  
    if size(test_inputs,1) > 0
        testLoss = loss(ann, test_inputs', test_targets')
        push!(testLosses, testLoss)
        # update message buffer
        log_message *= " - test_loss: $(round(testLoss, digits=4))"
        if showText
            # do nothing
            #println("Epoch ", numEpoch, ": test loss: ", testLoss)
        end
    end

    if showText
        # print message buffer
        println(join(log_message))
    end
    # --- Optimizer setup ---
    opt_state = Flux.setup(Adam(learningRate), ann)

    # --- Requirement: variables for early stopping ---
    epochsWithoutImprovement = 0
    bestValLoss = Inf
    bestAnn = deepcopy(ann)  # Requirement: store best ANN (deepcopy to avoid overwriting)
    bestAnnEpoch = 0

    while (numEpoch < maxEpochs) && (trainingLoss > minLoss) && (epochsWithoutImprovement < maxEpochsVal)
        Flux.train!(loss, ann, [(inputs', targets')], opt_state)
        numEpoch += 1
        log_message = []
        # --- Compute training loss and store it ---
        trainingLoss = loss(ann, inputs', targets')
        push!(trainingLosses, trainingLoss)
        
        # update message buffer
        log_message = "Epoch $numEpoch - loss: $(round(trainingLoss, digits=4))"

        outputs=ann(inputs')
        outputs=classifyOutputs(outputs')
        predicted_classes = Flux.onecold(outputs')        # vector of predicted labels
        true_classes = Flux.onecold(targets')      # vector of true labels

        accuracy_train=accuracy(predicted_classes, true_classes)

        log_message *= " - acc: $(round(accuracy_train, digits=4))"

        # --- Requirement: if validation set provided, track its loss for early stopping ---
        if size(val_inputs,1) > 0
            validationLoss = loss(ann, val_inputs', val_targets')
            push!(validationLosses, validationLoss)

            # update message buffer
            log_message *= " - val_loss: $(round(validationLoss, digits=4))"

            outputs=ann(val_inputs')
            outputs=classifyOutputs(outputs')
            predicted_classes = Flux.onecold(outputs')        # vector of predicted labels
            true_classes = Flux.onecold(val_targets')      # vector of true labels

            accuracy_val=accuracy(predicted_classes, true_classes)

            log_message *= " - val_acc: $(round(accuracy_val, digits=4))"

            if validationLoss < bestValLoss
                bestValLoss = validationLoss
                epochsWithoutImprovement = 0
                bestAnn = deepcopy(ann)   # Requirement: update best ANN when improvement found
                bestAnnEpoch = numEpoch
            else
                epochsWithoutImprovement += 1
            end
        end

        # --- Requirement: also track test loss if provided ---
        if size(test_inputs,1) > 0
            testLoss = loss(ann, test_inputs', test_targets')
            push!(testLosses, testLoss)
            

            # update message buffer
            log_message *= " - test_loss: $(round(testLoss, digits=4))"

            outputs=ann(test_inputs')
            outputs=classifyOutputs(outputs')
            predicted_classes = Flux.onecold(outputs')        # vector of predicted labels
            true_classes = Flux.onecold(test_targets')      # vector of true labels

            accuracy_test=accuracy(predicted_classes, true_classes)
            test_error = 1-accuracy_test

            log_message *= " - test_acc: $(round(accuracy_test, digits=4))"
            log_message *= " - test_error: $(round(test_error, digits=4))"

        end
        
        if showText
            # update message buffer
            log_message *= " - epochsWithoutImprovement $(epochsWithoutImprovement)"
            println(join(log_message))
        end

        #print("trainingLoss > minLoss : $(trainingLoss > minLoss) \n")
        #print("epochsWithoutImprovement < maxEpochsVal : $(epochsWithoutImprovement < maxEpochsVal) \n")
    end  # closes while
    
    # --- Early stopping notice ---
    if (epochsWithoutImprovement >= maxEpochsVal) && showText
        println("⏹ Early stopping triggered after $numEpoch epochs (no improvement for $maxEpochsVal epochs).")
    end

    # --- Requirement: return the right ANN ---
    # If validation set was provided → return best ANN found
    # Otherwise → return last trained ANN
    finalAnn = size(val_inputs,1) > 0 ? bestAnn : ann

    bestEpoch = size(val_inputs,1) > 0 ? bestAnnEpoch : maxEpochs
    println("The ANN corespond to the epoch $bestEpoch")

    return finalAnn, trainingLosses, validationLosses, testLosses
end  # closes function



function trainClassANN(topology::AbstractArray{<:Int,1},  
        trainingDataset::Tuple{AbstractArray{<:Real,2}, AbstractArray{Bool,1}}; 
        validationDataset::Tuple{AbstractArray{<:Real,2}, AbstractArray{Bool,1}}= 
                    (Array{eltype(trainingDataset[1]),2}(undef,0,0), falses(0)), 
        testDataset::Tuple{AbstractArray{<:Real,2}, AbstractArray{Bool,1}}= 
                    (Array{eltype(trainingDataset[1]),2}(undef,0,0), falses(0)), 
        transferFunctions::AbstractArray{<:Function,1}=fill(σ, length(topology)), 
        maxEpochs::Int=1000, minLoss::Real=0.0, learningRate::Real=0.01,  
        maxEpochsVal::Int=20, showText::Bool=false)

    (inputs, targets) = trainingDataset
    (val_inputs, val_targets) = validationDataset
    (test_inputs, test_targets) = testDataset
    
    # This function assumes that each sumple is in a row
    # we are going to check the numeber of samples to have same inputs and targets
    @assert(size(inputs,1)==size(targets,1));
    @assert (size(val_inputs,1) == size(val_targets,1));
    @assert (size(test_inputs,1) == size(test_targets,1));

    trainClassANN(topology, 
        (inputs, reshape(targets, length(targets), 1)),
        (val_inputs, reshape(val_targets, length(val_targets), 1)), 
        (test_inputs, reshape(test_targets, length(test_targets), 1)),
        transferFunctions, 
        maxEpochs=maxEpochs, minLoss=minLoss, learningRate=learningRate,
        maxEpochsVal, showText);
end;

function print_confusion_matrix(ann, dataset::Tuple{AbstractArray{<:Real,2}, AbstractArray{Bool,2}}, name::String)
    inputs, targets = dataset
    if size(inputs, 1) == 0
        println("No data for $name set.")
        return
    end

    outputs = ann(inputs')
    outputs = classifyOutputs(outputs')
    predicted_classes = Flux.onecold(outputs')
    true_classes = Flux.onecold(targets')

    # Build confusion matrix
    classes = sort(unique(true_classes))
    n = length(classes)
    cm = zeros(Int, n, n)
    for (a, p) in zip(true_classes, predicted_classes)
        i = findfirst(==(a), classes)
        j = findfirst(==(p), classes)
        cm[i, j] += 1
    end

    println("\nConfusion matrix for $name set (rows=actual, cols=predicted):")
    println("Classes: ", classes)
    println(cm)
end




print_confusion_matrix (generic function with 1 method)

In [67]:

const Losses = Flux
architectures = [
    [10, 5],
    [4, 8, 2],
    [6, 12, 6, 2] ,
    [16, 8, 4],
    [8, 4, 2, 2],
    [20, 10, 5, 2] , 
    [5,3]  
]

# --- Store all results ---
results = Dict{String, Tuple{Vector{Float32}, Vector{Float32}, Vector{Float32}}}()


for topology in architectures
    println("\nTraining architecture: ", topology)
    finalAnn, trainLoss, valLoss, testLoss = trainClassANN(
        topology,
        (trainingInputs, trainingTargets),
        validationDataset = (valInputs, valTargets),
        testDataset = (testInputs, testTargets),
        maxEpochs = 10,
        learningRate = 0.01,
        showText = true
    )
    results[string(topology)] = (trainLoss, valLoss, testLoss)
end


Training architecture: [10, 5]


MethodError: MethodError: no method matching trainClassANN(::Vector{Int64}, ::Tuple{Matrix{Float32}, Vector{Float32}}; validationDataset::Tuple{Matrix{Float32}, Vector{Float32}}, testDataset::Tuple{Matrix{Float32}, Vector{Float32}}, maxEpochs::Int64, learningRate::Float64, showText::Bool)
The function `trainClassANN` exists, but no method is defined for this combination of argument types.

Closest candidates are:
  trainClassANN(::Vector{Int64}, !Matched::Tuple{Matrix{Float32}, Matrix{Bool}}; validationDataset, testDataset, maxEpochs, learningRate, showText)
   @ Main e:\USC\master\courses\ML1\assignments\project\jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X25sZmlsZQ==.jl:18
  trainClassANN(::Vector{Int64}, !Matched::Tuple{Matrix{Float32}, Vector{Int64}}; validationDataset, transferFunctions, maxEpochs, learningRate, showText) got unsupported keyword argument "testDataset"
   @ Main e:\USC\master\courses\ML1\assignments\project\jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X1

In [69]:
# --- Helper function to print confusion matrix ---
function print_confusion_matrix(ann, dataset::Tuple{AbstractArray{<:Real,2}, AbstractArray{Bool,2}}, name::String)
    inputs, targets = dataset
    if size(inputs, 1) == 0
        println("No data for $name set.")
        return
    end

    outputs = ann(inputs')
    outputs = classifyOutputs(outputs')
    predicted_classes = Flux.onecold(outputs')
    true_classes = Flux.onecold(targets')

    # Build confusion matrix
    classes = sort(unique(true_classes))
    n = length(classes)
    cm = zeros(Int, n, n)
    for (a, p) in zip(true_classes, predicted_classes)
        i = findfirst(==(a), classes)
        j = findfirst(==(p), classes)
        cm[i, j] += 1
    end

    println("\nConfusion matrix for $name set (rows=actual, cols=predicted):")
    println("Classes: ", classes)
    println(cm)
end

# --- Loop over architectures and train ---
results = Dict{String, Tuple{Vector{Float32}, Vector{Float32}, Vector{Float32}}}()

for topology in architectures
    println("\nTraining architecture: ", topology)
    
    finalAnn, trainLoss, valLoss, testLoss = trainClassANN(
        topology,
        (trainingInputs, trainingTargets),
        validationDataset = (valInputs, valTargets),
        testDataset = (testInputs, testTargets),
        maxEpochs = 10,
        learningRate = 0.01,
        showText = true
    )
    
    # Store losses
    results[string(topology)] = (trainLoss, valLoss, testLoss)
    
    # Print confusion matrices
    print_confusion_matrix(finalAnn, (trainingInputs, trainingTargets), "Training")
    print_confusion_matrix(finalAnn, (valInputs, valTargets), "Validation")
    print_confusion_matrix(finalAnn, (testInputs, testTargets), "Test")
end



Training architecture: [10, 5]


MethodError: MethodError: no method matching trainClassANN(::Vector{Int64}, ::Tuple{Matrix{Float32}, Vector{Float32}}; validationDataset::Tuple{Matrix{Float32}, Vector{Float32}}, testDataset::Tuple{Matrix{Float32}, Vector{Float32}}, maxEpochs::Int64, learningRate::Float64, showText::Bool)
The function `trainClassANN` exists, but no method is defined for this combination of argument types.

Closest candidates are:
  trainClassANN(::Vector{Int64}, !Matched::Tuple{Matrix{Float32}, Matrix{Bool}}; validationDataset, testDataset, maxEpochs, learningRate, showText)
   @ Main e:\USC\master\courses\ML1\assignments\project\jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X25sZmlsZQ==.jl:18
  trainClassANN(::Vector{Int64}, !Matched::Tuple{Matrix{Float32}, Vector{Int64}}; validationDataset, transferFunctions, maxEpochs, learningRate, showText) got unsupported keyword argument "testDataset"
   @ Main e:\USC\master\courses\ML1\assignments\project\jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X1

In [68]:
# --- Plotting all losses one plot for each model---
using Plots
for (topology, (trainLoss, valLoss, testLoss)) in results
    epochs = 0:length(trainLoss)-1  # include cycle 0

    p = plot(epochs, trainLoss, label="Train Loss", lw=2, marker=:circle)
    
    if !isempty(valLoss)
        plot!(p, epochs, valLoss, label="Validation Loss", lw=2, linestyle=:dash, marker=:diamond)
    end
    
    if !isempty(testLoss)
        plot!(p, epochs, testLoss, label="Test Loss", lw=2, linestyle=:dot, marker=:star)
    end

    xlabel!("Epoch")
    ylabel!("Loss")
    title!("Loss Evolution - Topology: $(topology)")
    plot!(legend=:topright)
    
    display(p)  # show the plot
end

In [56]:
# --- Assume finalAnn is your trained network ---
# --- testInputs and testTargets are the test dataset ---

# Step 1: Get model outputs
outputs = finalAnn(testInputs')             # ANN expects features as columns

# Step 2: Convert outputs to boolean/multiclass predictions
predicted = classifyOutputs(outputs')       # rows = samples, columns = classes

# Step 3: Compute confusion matrix and metrics
accuracy, error_rate, sensitivity, specificity, ppv, npv, f1, cm = 
    confusionMatrix(predicted, testTargets)

# Step 4: Print results
println("Accuracy: ", round(accuracy, digits=4))
println("Error rate: ", round(error_rate, digits=4))
println("Sensitivity: ", round(sensitivity, digits=4))
println("Specificity: ", round(specificity, digits=4))
println("Precision (PPV): ", round(ppv, digits=4))
println("Negative Predictive Value (NPV): ", round(npv, digits=4))
println("F1 Score: ", round(f1, digits=4))
println("Confusion Matrix:")
println(cm)


UndefVarError: UndefVarError: `finalAnn` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [ ]:
using Flux
using Statistics
# 1.2 One-hot encode targets
targets_oh = oneHotEncoding(targets, classes)
println("One-hot encoded targets size: ", size(targets_oh))

trainingTargets = targets_oh[trainIdx, :]
valTargets   = targets_oh[valIdx, :]
testTargets  = targets_oh[testIdx, :]

println("Train set:  targets $(size(trainingTargets))")
println("Validation set:  targets $(size(valTargets))")
println("Test set: targets $(size(testTargets))")



architectures = [
    [10, 5],
    [4, 8, 2],
    [6, 12, 6, 2] ,
    [16, 8, 4],
    [8, 4, 2, 2],
    [20, 10, 5, 2] , 
    [5,3]  
]

# --- Store all results ---
results = Dict{String, Tuple{Vector{Float32}, Vector{Float32}, Vector{Float32}}}()
# --- Define loss function (binary or multi-class) ---
# discriminates based on the number of output neurons
loss(model, x, y) = (size(y,1) == 1) ? Flux.binarycrossentropy(model(x), y) : Flux.crossentropy(model(x), y)


for topology in architectures
    println("\nTraining architecture: ", topology)
    finalAnn, trainLoss, valLoss, testLoss = trainClassANN(
        topology,
        (trainingInputs, trainingTargets),
        validationDataset = (valInputs, valTargets),
        testDataset = (testInputs, testTargets),
        maxEpochs = 250,
        learningRate = 0.01,
        showText = true
    )
    results[string(topology)] = (trainLoss, valLoss, testLoss)
end


In [ ]:
using Flux
using Statistics

# --- One-hot encode targets for multi-class classification ---
targets_oh = oneHotEncoding(targets, classes)
println("One-hot encoded targets size: ", size(targets_oh))

trainingTargets = targets_oh[trainIdx, :]  # shape: (num_samples, num_classes)
valTargets      = targets_oh[valIdx, :]
testTargets     = targets_oh[testIdx, :]

println("Train set: targets $(size(trainingTargets))")
println("Validation set: targets $(size(valTargets))")
println("Test set: targets $(size(testTargets))")


# --- Training function ---
function trainClassANN(topology::Vector{Int},  
                       trainingDataset::Tuple{Matrix{Float32}, Matrix{Bool}}; 
                       validationDataset::Tuple{Matrix{Float32}, Matrix{Bool}}=(Float32[], Bool[]),
                       testDataset::Tuple{Matrix{Float32}, Matrix{Bool}}=(Float32[], Bool[]),
                       maxEpochs::Int=250,
                       learningRate::Float32=0.01,
                       showText::Bool=false)

    X_train, Y_train = trainingDataset
    X_val,   Y_val   = validationDataset
    X_test,  Y_test  = testDataset

    n_features = size(X_train, 2)
    n_classes  = size(Y_train, 2)

    # --- Build model with softmax output ---
    layers = []
    input_size = n_features
    for h in topology
        push!(layers, Dense(input_size, h, σ))
        input_size = h
    end
    push!(layers, Dense(input_size, n_classes))  # last layer before softmax
    model = Chain(layers..., softmax)

    # --- Loss function ---
    loss(x, y) = Flux.crossentropy(model(x), y)

    # --- Optimizer ---
    opt = Descent(learningRate)

    # --- Track losses ---
    train_losses = Float32[]
    val_losses   = Float32[]
    test_losses  = Float32[]

    for epoch in 1:maxEpochs
        gs = gradient(() -> loss(X_train', Y_train'), params(model))
        Flux.Optimise.update!(opt, params(model), gs)

        # Compute losses
        train_loss = loss(X_train', Y_train')
        push!(train_losses, train_loss)

        if size(X_val, 1) > 0
            val_loss = loss(X_val', Y_val')
            push!(val_losses, val_loss)
        end

        if size(X_test, 1) > 0
            test_loss = loss(X_test', Y_test')
            push!(test_losses, test_loss)
        end

        # Optional logging
        if showText && epoch % 10 == 0
            msg = "Epoch $epoch, train_loss: $(round(train_loss,digits=4))"
            if !isempty(val_losses)
                msg *= ", val_loss: $(round(val_losses[end], digits=4))"
            end
            if !isempty(test_losses)
                msg *= ", test_loss: $(round(test_losses[end], digits=4))"
            end
            println(msg)
        end
    end

    return model, train_losses, val_losses, test_losses
end


# --- Example architectures ---
architectures = [
    [10, 5],
    [4, 8, 2],
    [6, 12, 6, 2],
    [16, 8, 4],
    [8, 4, 2, 2],
    [20, 10, 5, 2],
    [5, 3]
]

results = Dict{String, Tuple{Vector{Float32}, Vector{Float32}, Vector{Float32}}}()

for topology in architectures
    println("\nTraining architecture: ", topology)
    finalAnn, trainLoss, valLoss, testLoss = trainClassANN(
        topology,
        (trainingInputs, trainingTargets),
        validationDataset = (valInputs, valTargets),
        testDataset       = (testInputs, testTargets),
        maxEpochs = 250,
        learningRate = 0.01f0,
        showText = true
    )
    results[string(topology)] = (trainLoss, valLoss, testLoss)
end
